# EcoSort Waste Management Assistant

**Module 8 Summative Lab**

**Datasets used**

RealWaste (UCI ML Repository, id 908), an image classification dataset with 9 waste material classes, `waste_descriptions.csv`, 5,000 text descriptions of waste items, and `waste_policy_documents.json`, 14 Metro City recycling policy documents.

**Environment note**

This notebook is designed to run on either Kaggle or locally. On Kaggle, attach the RealWaste dataset and the ecosort data dataset (containing the CSV and JSON files) as data sources, they mount automatically under `/kaggle/input/`. Locally, place the extracted files in the same directory as this notebook. A GPU is strongly recommended for the CNN training section.

---

In [ ]:
# Core imports, warnings suppressed for a cleaner notebook read
import warnings
warnings.filterwarnings('ignore')

import os
import json
import random
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, UnidentifiedImageError

import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.applications import MobileNetV2, EfficientNetB0
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics.pairwise import cosine_similarity

# reproducibility, matches the starter notebook's seed setup
np.random.seed(42)
tf.random.set_seed(42)
random.seed(42)

## 1. Business Understanding

**Scenario**

EcoSort is building an intelligent waste management assistant for Metro City's waste management department. Residents currently are not sure how to sort waste correctly, which slows down manual sorting at intake facilities. The assistant needs to shortcut that uncertainty at the point residents are deciding what to do with an item.

**Problem statement**

Metro City needs a system that can take either a photo of a waste item or a short text description of it, work out which of the 9 material categories it belongs to, and return specific, policy grounded disposal instructions, without a resident needing to know the local recycling rules themselves.

**Objectives**

1. Classify waste items from images into 9 material categories, using a CNN
2. Classify waste items from short text descriptions into the same 9 categories, using a text classifier
3. Retrieve and generate accurate, policy grounded disposal instructions for a given category, using RAG over the Metro City policy documents
4. Integrate all three into a single assistant that accepts either an image or a text description as input

**Success Criteria**

- Image classifier and text classifier both generalize to held-out test data, not just training data
- Generated recycling instructions are grounded in the actual policy documents (not hallucinated) and cite the relevant policy
- The integrated assistant produces a category + confidence + instructions for both input types

## 2. Data Understanding

Before any preprocessing, this section looks at all three data sources as they are, how large each one is, how the 9 categories are distributed, and what shape the raw data comes in. Those observations drive the preparation decisions in Section 3.

### 2.1 RealWaste Image Dataset

In [ ]:

realwaste_path = kagglehub.dataset_download("kevinkiplangat432/realwaste")
print("RealWaste path:", realwaste_path)
for item in os.listdir(realwaste_path):
    print(" ", item)

print()

ecosort_data_path = kagglehub.dataset_download("kevinkiplangat432/ecosort-data")
print("EcoSort Data path:", ecosort_data_path)
for item in os.listdir(ecosort_data_path):
    print(" ", item)

In [ ]:
# Detect environment automatically, local vs Kaggle attached datasets.
# Multiple candidates checked since Kaggle's mount structure can vary.
local_path = Path('RealWaste')
kaggle_candidates = [
    Path('/kaggle/input/datasets/kevinkiplangat432/realwaste'),
    Path('/kaggle/input/realwaste/RealWaste'),
    Path('/kaggle/input/realwaste'),
]

data_dir = None
for candidate in kaggle_candidates:
    if candidate.exists() and any(d.is_dir() for d in candidate.glob('*')):
        data_dir = candidate
        break

if data_dir is None and local_path.exists():
    data_dir = local_path

if data_dir is None:
    raise FileNotFoundError(
        "RealWaste folder not found locally or on Kaggle. "
        "Attach the dataset (Kaggle) or place the extracted 'RealWaste' folder next to this notebook (local)."
    )

class_names_found = sorted([d.name for d in data_dir.glob('*') if d.is_dir()])
image_counts = {c: len(list((data_dir / c).glob('*.jpg'))) for c in class_names_found}
total_images = sum(image_counts.values())

In [ ]:
print(f"Data directory in use, {data_dir}")
print(f"Number of classes, {len(class_names_found)}")
print(f"Class names, {class_names_found}")
print(f"Total images found, {total_images}")
print()
for c, n in image_counts.items():
    print(f"{c}, {n} images")

Per the UCI dataset page, this should resolve to 4,752 images across 9 classes. Cardboard 461, Food Organics 411, Glass 420, Metal 790, Miscellaneous Trash 495, Paper 500, Plastic 921, Textile Trash 318, Vegetation 436. The spread is uneven, Plastic and Metal together make up roughly 36% of the dataset, while Textile Trash has under a third as many images as Plastic. This imbalance is accounted for in Section 3 through class weighting.

A note on data integrity, an earlier pass over this dataset found that 3,146 of 4,752 files were unreadable, concentrated almost entirely in 6 of the 9 classes, traced back to an interrupted zip extraction rather than scattered individual corruption. A clean re download resolved it. This is worth checking early with any large downloaded dataset, since a corrupted extraction can silently gut specific classes rather than fail loudly.

In [ ]:
# Class balance visualisation
plt.figure(figsize=(9, 5))
plt.bar(image_counts.keys(), image_counts.values(), color='seagreen')
plt.xlabel('Category')
plt.ylabel('Number of Images')
plt.title('RealWaste, Image Count per Class')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Full corrupt file scan, run once, quarantine anything unreadable
def scan_for_corrupt_images(data_dir, class_names):
    corrupt_files = []
    total_checked = 0
    for c in class_names:
        for f in (data_dir / c).glob('*.jpg'):
            total_checked += 1
            try:
                with Image.open(f) as img:
                    img.load()
            except Exception as e:
                corrupt_files.append((str(f), type(e).__name__))
    print(f"Checked {total_checked} images")
    print(f"Corrupt or unreadable, {len(corrupt_files)}")
    return corrupt_files

corrupt_files = scan_for_corrupt_images(data_dir, class_names_found)
for f, err in corrupt_files[:20]:
    print(f"  {f}, {err}")

`[FILL IN AFTER KAGGLE RUN]` Confirm the corrupt file count on the dataset actually attached to this run, expected to be 0 given the clean re download, but worth verifying since Kaggle mounted datasets are a fresh copy.

In [ ]:
# Sample a handful of images per class to check raw resolution consistency
sample_dims = set()
for c in class_names_found:
    files = list((data_dir / c).glob('*.jpg'))[:10]
    for f in files:
        with Image.open(f) as img:
            sample_dims.add(img.size)

print(f"Unique image dimensions found in sample, {sample_dims}")

UCI documents these as 524x524 releases, so this cell is mainly a sanity check that the attached copy matches that.

### 2.2 Waste Description Text Data

In [ ]:
# CSV path, local vs Kaggle
csv_local = Path('waste_descriptions.csv')
csv_kaggle = Path('/kaggle/input/ecosort-data/waste_descriptions.csv')
csv_path = csv_kaggle if csv_kaggle.exists() else csv_local

desc_df = pd.read_csv(csv_path)
print(f"Shape, {desc_df.shape}")
print(f"Columns, {list(desc_df.columns)}")
desc_df.head()

In [ ]:
print("Missing values per column,")
print(desc_df.isna().sum())
print()
print("Category distribution,")
print(desc_df['category'].value_counts())

In [ ]:
plt.figure(figsize=(9, 5))
desc_df['category'].value_counts().plot(kind='bar', color='steelblue')
plt.xlabel('Category')
plt.ylabel('Number of Descriptions')
plt.title('Waste Descriptions, Count per Category')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
desc_lengths = desc_df['description'].str.split().str.len()
print(desc_lengths.describe())

5,000 rows, 9 categories, no missing values in `description`, `category`, or `material_composition`. `common_confusion` is null for about half the rows, 2,504 of 5,000, expected since it is presumably only populated for items that are genuinely easy to mis sort. Class balance is close, counts range from 506 to 600, nowhere near as skewed as the image dataset. Descriptions are short, a handful of words each, median around 5 to 6 words, so this points toward a lightweight text classifier, TF IDF plus classical ML, rather than a heavier transformer.

### 2.3 Waste Policy Documents

In [ ]:
# JSON path, local vs Kaggle
json_local = Path('waste_policy_documents.json')
json_kaggle = Path('/kaggle/input/ecosort-data/waste_policy_documents.json')
json_path = json_kaggle if json_kaggle.exists() else json_local

with open(json_path) as f:
    policies = json.load(f)

print(f"Number of policy documents, {len(policies)}")
print(f"Fields per document, {list(policies[0].keys())}")

In [ ]:
cats_covered = Counter()
for p in policies:
    for c in p['categories_covered']:
        cats_covered[c] += 1

print("Documents covering each category,")
for c, n in sorted(cats_covered.items(), key=lambda x, n=None: -x[1]):
    print(f"{c}, {n} document(s)")

single_cat = sum(1 for p in policies if len(p['categories_covered']) == 1)
multi_cat = len(policies) - single_cat
print(f"\nSingle category documents, {single_cat}")
print(f"Multi category documents, {multi_cat}")

14 policy documents total. Every one of the 9 categories has at least one dedicated single category document, and 5 broader documents each bundle several categories together and repeat guideline text verbatim from the single category docs. That repetition matters for the RAG step in Section 3, a naive retriever could return two documents saying nearly the same thing for one category, so retrieval either dedupes by category or ranks the single category canonical document above the bundled ones.

In [ ]:
print(policies[0]['document_text'][:400])

Each document is unstructured plain text organised into consistent subsections, Acceptable Items, Non Acceptable Items, Collection Method, Preparation Instructions, Benefits. That consistent structure is useful, chunking by these labeled subsections rather than fixed token windows keeps retrieved chunks coherent.

### Section Conclusion

All three data sources are in reasonable shape, no missing critical fields, and category taxonomies match exactly across the image dataset, the text descriptions, and the policy documents' `categories_covered` field, all use the same 9 labels. The main issue carried into Data Preparation is class imbalance in the image data, Plastic is roughly 3 times Textile Trash, which the text data does not share. This means the CNN training step needs class weights, while the text classifier training does not.

## 3. Data Preparation

This section builds the three input pipelines the modelling sections rely on, an image pipeline for the CNN, a text pipeline for the description classifier, and a document retrieval pipeline for the RAG system.

### 3.1 Image Data Pipeline

The RealWaste images are split into training, validation, and test sets. Class weights are computed here too, since Section 2 showed a meaningful imbalance that the CNN needs to account for during training.

In [ ]:
BATCH_SIZE = 32
IMG_HEIGHT = 224
IMG_WIDTH = 224

# int labels, not one hot, so class_weight can be used directly in model.fit()
train_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    label_mode='int',
    shuffle=True
)

validation_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    label_mode='int',
    shuffle=True
)

In [ ]:
print(f"Training batches, {tf.data.experimental.cardinality(train_ds)}")
print(f"Validation batches, {tf.data.experimental.cardinality(validation_ds)}")

An 80/20 split gives the training set the bulk of the data while holding back a meaningful validation set. The validation set still needs splitting further to carve out a genuine test set, held completely separate from any training decision, that happens next.

In [ ]:
val_batches = tf.data.experimental.cardinality(validation_ds)
test_dataset = validation_ds.take(val_batches // 2)
validation_ds = validation_ds.skip(val_batches // 2)

In [ ]:
print(f"Validation batches, {tf.data.experimental.cardinality(validation_ds)}")
print(f"Test batches, {tf.data.experimental.cardinality(test_dataset)}")

Splitting the validation set in half keeps the test set completely untouched by any training time decision, early stopping, learning rate, architecture tweaks, while validation still has enough batches left to be a meaningful signal during training.

In [ ]:
# Class weights, counters minority class underrepresentation
class_names = train_ds.class_names

counts_ordered = [image_counts[c] for c in class_names]
class_weights_array = compute_class_weight(
    class_weight='balanced',
    classes=np.arange(len(class_names)),
    y=np.repeat(np.arange(len(class_names)), counts_ordered)
)
class_weights = dict(enumerate(class_weights_array))

In [ ]:
for i, c in enumerate(class_names):
    print(f"{c}, weight = {class_weights[i]:.3f}")

Textile Trash, the smallest class, gets the highest weight, while Plastic and Metal get weights below 1, since they are already overrepresented. Passing `class_weights` into `model.fit()` in Section 6 means misclassifying a minority class image costs the loss function more than misclassifying a majority class one.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
validation_ds = validation_ds.cache().prefetch(buffer_size=AUTOTUNE)
test_dataset = test_dataset.cache().prefetch(buffer_size=AUTOTUNE)

Caching avoids re reading images from disk on every epoch, and prefetching overlaps data loading with model computation so the GPU is not left idle.

### 3.2 Text Data Pipeline

The waste description data is balanced across categories, so no class weighting is needed here, unlike the image pipeline. Descriptions are short, so a bag of words style representation, TF IDF, captures enough signal without needing a heavier transformer tokeniser.

In [ ]:
X = desc_df['description']
y = desc_df['category']

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

X_train_text, X_test_text, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

In [ ]:
print(f"Training samples, {len(X_train_text)}")
print(f"Test samples, {len(X_test_text)}")
print(f"Classes, {list(label_encoder.classes_)}")

`stratify=y_encoded` keeps the 80/20 split proportional across all 9 categories, so the test set mirrors the near even class balance seen in the full dataset.

In [ ]:
vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words='english',
    ngram_range=(1, 2),
    max_features=2000
)

X_train_tfidf = vectorizer.fit_transform(X_train_text)
X_test_tfidf = vectorizer.transform(X_test_text)

In [ ]:
print(f"Training feature matrix shape, {X_train_tfidf.shape}")
print(f"Test feature matrix shape, {X_test_tfidf.shape}")
print(f"Vocabulary size, {len(vectorizer.vocabulary_)}")

Fitting the vectoriser only on the training text avoids leaking test set vocabulary into the feature space. `ngram_range=(1,2)` captures short phrases like "food residue" that carry more signal than either word alone, given how short these descriptions are.

### 3.3 RAG Document Preparation

Section 2.3 found that policy documents share a consistent internal structure and that some documents bundle multiple categories together, repeating text that already appears in a dedicated single category document. This section chunks documents by their labeled subsections rather than fixed token windows, and prioritises single category canonical documents over bundled ones.

In [ ]:
import re

def chunk_policy_document(policy):
    """Split a policy document into labeled subsection chunks."""
    text = policy['document_text']
    sections = re.split(r'\n(?=[A-Z][A-Za-z ]+:?\n)', text)

    chunks = []
    for section in sections:
        section = section.strip()
        if not section:
            continue
        chunks.append({
            'policy_id': policy['policy_id'],
            'policy_type': policy['policy_type'],
            'categories_covered': policy['categories_covered'],
            'is_canonical': len(policy['categories_covered']) == 1,
            'text': section
        })
    return chunks

all_chunks = []
for policy in policies:
    all_chunks.extend(chunk_policy_document(policy))

In [ ]:
print(f"Total chunks created, {len(all_chunks)}")
print(f"From {len(policies)} source documents")
print()
print("Example chunk,")
print(all_chunks[0])

Splitting on the header pattern breaks each document into its natural subsections rather than an arbitrary token window, so a retrieved chunk stays coherent. The `is_canonical` flag marks chunks from single category documents, used for retrieval ranking next.

In [ ]:
from sentence_transformers import SentenceTransformer

# CPU explicitly, avoids GPU kernel mismatches seen on some Kaggle accelerator builds
embedder = SentenceTransformer('all-MiniLM-L6-v2', device='cpu')

chunk_texts = [c['text'] for c in all_chunks]
chunk_embeddings = embedder.encode(chunk_texts, show_progress_bar=True, device='cpu')

In [ ]:
print(f"Embedding matrix shape, {chunk_embeddings.shape}")

`all-MiniLM-L6-v2` is a lightweight sentence embedding model, fast enough to run on CPU, which matters since this notebook is already GPU bound by the CNN.

In [ ]:
def retrieve_policy_chunks(query, top_k=3, prefer_canonical=True):
    """Open retrieval across all chunks. Kept for general purpose search,
    the category filtered version below is preferred once a category is known."""
    query_embedding = embedder.encode([query], device='cpu')
    similarities = cosine_similarity(query_embedding, chunk_embeddings)[0]

    results = []
    for i, sim in enumerate(similarities):
        chunk = all_chunks[i]
        score = sim + (0.05 if prefer_canonical and chunk['is_canonical'] else 0)
        results.append((score, chunk))

    results.sort(key=lambda x: x[0], reverse=True)
    return [r[1] for r in results[:top_k]]

In [ ]:
def retrieve_policy_chunks_for_category(category, query=None, top_k=3):
    """Retrieve chunks restricted to documents that cover the known category.
    This is the primary retrieval path once a category has already been
    classified, since it removes the chance of an unrelated document being
    retrieved purely on loose semantic similarity. Falls back to open
    search only if no documents mention the category at all."""
    category_chunks = [
        (i, c) for i, c in enumerate(all_chunks)
        if category in c['categories_covered']
    ]

    if not category_chunks:
        return retrieve_policy_chunks(query or category, top_k=top_k)

    search_text = query if query else f"disposal instructions for {category}"
    query_embedding = embedder.encode([search_text], device='cpu')

    indices = [i for i, c in category_chunks]
    chunks_only = [c for i, c in category_chunks]
    category_embeddings = chunk_embeddings[indices]

    similarities = cosine_similarity(query_embedding, category_embeddings)[0]

    scored = []
    for sim, chunk in zip(similarities, chunks_only):
        score = sim + (0.05 if chunk['is_canonical'] else 0)
        scored.append((score, chunk))

    scored.sort(key=lambda x: x[0], reverse=True)
    return [c for _, c in scored[:top_k]]

In [ ]:
# quick sanity check, open retrieval
test_results = retrieve_policy_chunks("plastic bottle recycling")
for r in test_results:
    print(f"[{r['policy_type']}] canonical={r['is_canonical']}")
    print(r['text'][:150])
    print()

The small canonical document bonus, plus 0.05 similarity, means that for a query like "plastic bottle recycling," the dedicated Plastic Recycling Guidelines document is favoured over the same content appearing inside a bundled document, even when both would otherwise score similarly on pure semantic similarity.

Note however that open, unfiltered retrieval like this is only used as a fallback. Section 11 uses `retrieve_policy_chunks_for_category` instead once a category is already known from classification, which removes the risk of an unrelated document being retrieved on loose semantic similarity alone, the exact failure mode found during initial testing where a query about batteries retrieved Vegetation guidelines.

## 4. Exploratory Data Analysis

Section 2 examined the raw data before any processing. This section looks at the data as it will actually reach the models, sample batches from the image pipeline, and a look at what the TF IDF vectoriser picked up as the most informative terms per category.

In [ ]:
plt.figure(figsize=(10, 10))
for images, labels in train_ds.take(1):
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(class_names[labels[i]])
        plt.axis("off")
plt.show()

Nine images pulled directly from the training pipeline, each labeled with its class. Worth checking these look sensible, correct orientation, label matches the visible object, before any time is spent training on top of them.

In [ ]:
for images, labels in train_ds.take(1):
    print(f"Batch image shape, {images.shape}")
    print(f"Batch label shape, {labels.shape}")
    print(f"Pixel value range, {images.numpy().min()} to {images.numpy().max()}")

Confirms images are batched at 224x224x3 as configured in Section 3.1, and pixel values still sit in the raw 0 to 255 range, normalisation has not been applied yet since it is handled inside the model itself in Section 5.

In [ ]:
feature_names = vectorizer.get_feature_names_out()

for i, category in enumerate(label_encoder.classes_):
    class_mask = y_train == i
    if class_mask.sum() == 0:
        continue
    avg_tfidf = X_train_tfidf[class_mask].mean(axis=0).A1
    top_indices = avg_tfidf.argsort()[-5:][::-1]
    top_terms = [feature_names[j] for j in top_indices]
    print(f"{category}, {', '.join(top_terms)}")

These are the terms carrying the most weight for each category on average, a rough sanity check that the vectoriser is picking up genuinely category specific vocabulary rather than generic filler.

## 5. Modelling

This section builds a CNN using transfer learning rather than training from scratch, given the RealWaste dataset's moderate size. MobileNetV2, pretrained on ImageNet, provides a feature extractor already tuned on millions of general images.

In [ ]:
base_model = MobileNetV2(
    input_shape=(IMG_HEIGHT, IMG_WIDTH, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False  # frozen initially, unfrozen later for fine tuning in Section 8

model = Sequential([
    layers.Rescaling(1./255, input_shape=(IMG_HEIGHT, IMG_WIDTH, 3)),
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.3),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(len(class_names), activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
model.summary()

Freezing `base_model` means MobileNetV2's pretrained weights are not touched at first, only the new head trains initially. Two dropout layers guard against overfitting on a relatively small dataset. `sparse_categorical_crossentropy` is used because the labels are integer encoded, not one hot, which is also what makes `class_weight` usable directly in `model.fit()` next.

### Discussion

**Why transfer learning instead of training from scratch**

A CNN trained from scratch on under 5,000 images would struggle to learn generalisable low level features, edges, textures, colours, from that little data alone. MobileNetV2's ImageNet pretraining already encodes those general visual features from over a million images, only the task specific head needs to learn from RealWaste. MobileNetV2 is also lightweight relative to larger nets, which matters for keeping training fast on a shared GPU quota.

## 6. Train the Model

The model built in Section 5 is trained here using the class weights computed in Section 3.1. EarlyStopping and ModelCheckpoint guard against overfitting and preserve the best epoch.

In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

checkpoint = ModelCheckpoint(
    'best_cnn_model.keras',
    monitor='val_loss',
    save_best_only=True
)

In [ ]:
EPOCHS = 20

history = model.fit(
    train_ds,
    validation_data=validation_ds,
    epochs=EPOCHS,
    class_weight=class_weights,
    callbacks=[early_stop, checkpoint]
)

In [ ]:
final_epoch = len(history.history['loss'])
print(f"Training stopped after {final_epoch} epochs, out of {EPOCHS} max")
print(f"Best validation loss, {min(history.history['val_loss']):.4f}")
print(f"Best validation accuracy, {max(history.history['val_accuracy']):.4f}")

`[FILL IN AFTER KAGGLE RUN]` Confirm the epoch count and best validation metrics match this run, note whether training stopped early or ran the full 20 epochs.

## 7. Learning Curves

Plotting training and validation loss and accuracy across epochs helps confirm the model is learning generalisable patterns rather than memorising the training set.

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training vs Validation Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Training vs Validation Accuracy')
plt.legend()

plt.tight_layout()
plt.show()

`[FILL IN AFTER KAGGLE RUN]` Describe what the curves show, whether validation loss tracked training loss closely, diverged early suggesting overfitting, or both stayed high suggesting underfitting.

### Section Conclusion

`[FILL IN AFTER KAGGLE RUN]` Summarise whether the baseline model is ready for evaluation as is, before moving to fine tuning below.

## 8. Evaluation

The model is evaluated here against `test_dataset`, the portion of data held back in Section 3.1 that was never touched during training or validation.

In [ ]:
test_loss, test_accuracy = model.evaluate(test_dataset)

In [ ]:
print(f"Test loss, {test_loss:.4f}")
print(f"Test accuracy, {test_accuracy:.4f}")

In [ ]:
y_true = []
y_pred = []

for images, labels in test_dataset:
    preds = model.predict(images, verbose=0)
    y_pred.extend(np.argmax(preds, axis=1))
    y_true.extend(labels.numpy())

print(classification_report(y_true, y_pred, target_names=class_names))

In [ ]:
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(9, 8))
plt.imshow(cm, cmap='Blues')
plt.colorbar()
plt.xticks(range(len(class_names)), class_names, rotation=45, ha='right')
plt.yticks(range(len(class_names)), class_names)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix, CNN Test Set, Baseline')
plt.tight_layout()
plt.show()

`[FILL IN AFTER KAGGLE RUN]` Identify which classes the baseline model confuses most, and whether Textile Trash, the smallest class, still underperforms despite class weighting.

### Fine Tuning the Model

The baseline above only trains a new head on top of a fully frozen MobileNetV2. This step unfreezes the top layers of the base model and retrains at a low learning rate, letting the pretrained features adjust slightly toward waste specific imagery rather than staying fixed at purely generic ImageNet features.

In [ ]:
# Unfreeze the top layers of the base model only, the earliest layers stay
# frozen since they encode generic low level features, edges and textures,
# that transfer well regardless of the target task
base_model.trainable = True

fine_tune_at = len(base_model.layers) - 30
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),  # much lower than the default, avoids destroying pretrained weights
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
FINE_TUNE_EPOCHS = 10

fine_tune_history = model.fit(
    train_ds,
    validation_data=validation_ds,
    epochs=FINE_TUNE_EPOCHS,
    class_weight=class_weights,
    callbacks=[early_stop, checkpoint]
)

In [ ]:
fine_tune_test_loss, fine_tune_test_accuracy = model.evaluate(test_dataset)
print(f"Fine tuned test loss, {fine_tune_test_loss:.4f}")
print(f"Fine tuned test accuracy, {fine_tune_test_accuracy:.4f}")
print(f"Baseline test accuracy was, {test_accuracy:.4f}")

In [ ]:
y_true_ft = []
y_pred_ft = []

for images, labels in test_dataset:
    preds = model.predict(images, verbose=0)
    y_pred_ft.extend(np.argmax(preds, axis=1))
    y_true_ft.extend(labels.numpy())

print(classification_report(y_true_ft, y_pred_ft, target_names=class_names))

`[FILL IN AFTER KAGGLE RUN]` Compare fine tuned accuracy against the frozen baseline directly, state whether unfreezing the top 30 layers improved generalisation, and note any class where fine tuning helped or hurt most.

### Section Conclusion

`[FILL IN AFTER KAGGLE RUN]` State the final test accuracy, baseline versus fine tuned, and confirm which version is carried forward into Section 12's integrated assistant.

## 9. Text Classifier Modelling and Training

Unlike the CNN, this classifier trains in seconds rather than needing GPU time, so modelling and training are combined into one section. Logistic Regression is used as the primary model, since it pairs well with TF IDF features, trains fast, and its coefficients are interpretable.

In [ ]:
text_classifier = LogisticRegression(
    max_iter=1000,
    random_state=42
)

text_classifier.fit(X_train_tfidf, y_train)

In [ ]:
train_accuracy = text_classifier.score(X_train_tfidf, y_train)
print(f"Training accuracy, {train_accuracy:.4f}")

No class weighting here, unlike the CNN, since Section 2.2 confirmed the text data is already close to balanced across categories. Training accuracy alone does not confirm the model generalises, that gets checked against the held out test set next.

## 10. Text Classifier Evaluation

Same evaluation approach as the CNN in Section 8, applied here to the text classifier's held out test set.

In [ ]:
y_pred_text = text_classifier.predict(X_test_tfidf)

print(classification_report(y_test, y_pred_text, target_names=label_encoder.classes_))

In [ ]:
cm_text = confusion_matrix(y_test, y_pred_text)

plt.figure(figsize=(9, 8))
plt.imshow(cm_text, cmap='Greens')
plt.colorbar()
plt.xticks(range(len(label_encoder.classes_)), label_encoder.classes_, rotation=45, ha='right')
plt.yticks(range(len(label_encoder.classes_)), label_encoder.classes_)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix, Text Classifier Test Set')
plt.tight_layout()
plt.show()

In [ ]:
feature_names = vectorizer.get_feature_names_out()

for i, category in enumerate(label_encoder.classes_):
    top_indices = text_classifier.coef_[i].argsort()[-5:][::-1]
    top_terms = [feature_names[j] for j in top_indices]
    print(f"{category}, strongest terms, {', '.join(top_terms)}")

`[FILL IN AFTER KAGGLE RUN]` State the test accuracy and whether it clears the CNN's accuracy, expected given how much cleaner the text signal is compared to raw images. If accuracy is very close to perfect, note that as worth a caveat rather than an unqualified positive, since it may indicate the category defining vocabulary is strongly present in the description text itself, a genuinely easier task rather than a flaw, but worth stating plainly rather than presenting as a like for like comparison against the CNN's harder image task.

### Section Conclusion

`[FILL IN AFTER KAGGLE RUN]` Compare CNN accuracy versus text classifier accuracy directly, note that the assistant in Section 12 routes between these two models depending on whether the resident provides an image or a description.

## 11. RAG System

Retrieval was built in Section 3.3, chunking, embeddings, and category filtered ranking. This section adds the generation step, turning retrieved chunks into a natural language answer grounded in the actual policy text.

**Note on an earlier retrieval design**

An initial version of this system used open, unfiltered semantic search across all 14 policy documents for every query, regardless of whether the waste category was already known. Testing surfaced two real grounding failures under that design. A query about a broken drinking glass returned the Acceptable Items line from Glass Recycling Guidelines instead of the Non Acceptable Items line that actually excludes drinking glasses, and a query about batteries retrieved Vegetation, Metal, and Cardboard guidelines, missing the correct Miscellaneous Trash document entirely, since the query wording did not closely match the policy text.

The fix implemented here is structural rather than just a bigger model. By the time this system is called from Section 12, the category is already known from the CNN or text classifier, so retrieval is restricted to only the documents covering that category, `retrieve_policy_chunks_for_category` from Section 3.3, rather than searching openly across all 14 documents. This removes the possibility of an unrelated category's document being retrieved at all. The generation model was also upgraded from flan t5 small to flan t5 base, and decoding switched from greedy to beam search, both allowed adjustments under the lab's own instructions to select an appropriate language model and adjust sampling methods and parameters.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# flan t5 base, upgraded from small for better factual extraction from context
tokenizer = AutoTokenizer.from_pretrained('google/flan-t5-base')
seq2seq_model = AutoModelForSeq2SeqLM.from_pretrained('google/flan-t5-base')
seq2seq_model = seq2seq_model.to('cpu')  # CPU explicitly, avoids GPU kernel mismatches seen on some Kaggle builds

def generate_text(prompt, max_length=200):
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True)
    outputs = seq2seq_model.generate(
        **inputs,
        max_length=max_length,
        num_beams=4,
        early_stopping=True
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
def generate_disposal_answer(category, query=None, top_k=3):
    """Generate grounded disposal instructions for a known waste category.
    Retrieval is restricted to documents covering this category."""
    retrieved = retrieve_policy_chunks_for_category(category, query=query, top_k=top_k)
    context = "\n\n".join([r['text'] for r in retrieved])

    prompt = (
        f"Based only on the following official policy text, give specific "
        f"disposal instructions for {category}. Include what is accepted, "
        f"what is not accepted, and how to prepare it.\n\n"
        f"Policy text,\n{context}\n\n"
        f"Instructions,"
    )

    response = generate_text(prompt)

    return {
        'category': category,
        'answer': response,
        'sources': list(set(r['policy_type'] for r in retrieved))  # deduped, avoids listing the same document repeatedly
    }

In [ ]:
# test against the categories that previously showed grounding failures
test_categories = ["Glass", "Miscellaneous Trash", "Plastic"]

for cat in test_categories:
    result = generate_disposal_answer(cat)
    print(f"Category, {result['category']}")
    print(f"Answer, {result['answer']}")
    print(f"Sources, {', '.join(result['sources'])}")
    print()

`[FILL IN AFTER KAGGLE RUN]` Confirm the Glass answer now correctly distinguishes accepted glass containers from excluded items like drinking glasses, and confirm the Miscellaneous Trash answer correctly surfaces the batteries special handling guidance rather than an unrelated category's document. State plainly whether the category filtered retrieval and flan t5 base upgrade resolved both grounding failures found in initial testing, and if either persists, note it honestly rather than treating the fix as fully complete.

### Section Conclusion

`[FILL IN AFTER KAGGLE RUN]` State whether the RAG system reliably grounds its answers in the correct policy document following the fix, and confirm it is ready to be wired into the integrated assistant in Section 12.

## 12. Integrated Assistant

This section ties Sections 5 through 11 together into a single entry point. A resident provides either an image or a text description, the assistant routes it to the matching classifier, and the predicted category feeds directly into the category filtered RAG system from Section 11 to return grounded disposal instructions.

In [ ]:
def classify_image(image_path):
    """Classify a waste item from an image file path."""
    img = tf.keras.utils.load_img(image_path, target_size=(IMG_HEIGHT, IMG_WIDTH))
    img_array = tf.keras.utils.img_to_array(img)
    img_array = tf.expand_dims(img_array, 0)

    predictions = model.predict(img_array, verbose=0)
    predicted_index = np.argmax(predictions[0])
    confidence = float(predictions[0][predicted_index])

    return {
        'category': class_names[predicted_index],
        'confidence': confidence
    }

In [ ]:
def classify_text(description):
    """Classify a waste item from a text description."""
    vectorized = vectorizer.transform([description])
    predicted_index = text_classifier.predict(vectorized)[0]
    confidence = float(max(text_classifier.predict_proba(vectorized)[0]))

    return {
        'category': label_encoder.inverse_transform([predicted_index])[0],
        'confidence': confidence
    }

In [ ]:
def ecosort_assistant(image_path=None, description=None):
    """
    Full EcoSort pipeline. Accepts either an image path or a text
    description, classifies the waste category, and returns grounded
    disposal instructions. The predicted category is passed straight
    into the category filtered RAG retrieval, not a free text query,
    since the category is already known with confidence at this point.
    """
    if image_path is not None:
        classification = classify_image(image_path)
        input_type = 'image'
    elif description is not None:
        classification = classify_text(description)
        input_type = 'text'
    else:
        raise ValueError("Provide either image_path or description")

    category = classification['category']
    rag_result = generate_disposal_answer(category=category)

    return {
        'input_type': input_type,
        'predicted_category': category,
        'confidence': classification['confidence'],
        'disposal_instructions': rag_result['answer'],
        'policy_sources': rag_result['sources']
    }

In [ ]:
# text smoke test, image test needs an actual file path from the test set
test_result = ecosort_assistant(description="crumpled aluminum can with some soda residue")
print(f"Input type, {test_result['input_type']}")
print(f"Predicted category, {test_result['predicted_category']}")
print(f"Confidence, {test_result['confidence']:.2f}")
print(f"Instructions, {test_result['disposal_instructions']}")
print(f"Sources, {', '.join(test_result['policy_sources'])}")

`[FILL IN AFTER KAGGLE RUN]` Confirm the end to end pipeline works correctly, predicted category is sensible for the test input, confidence score is reasonably high, and the returned disposal instructions match what the policy documents actually say for that category. Also test at least one image input directly against a sample file from the RealWaste test set to confirm both entry points work.

### Section Conclusion

`[FILL IN AFTER KAGGLE RUN]` State whether all four Section 1 objectives are met by this integrated function, classify from image, classify from text, retrieve grounded instructions, single entry point for both input types.

## 13. Deployment

A full production deployment is out of scope for this lab, but a simple interactive interface demonstrates the assistant working end to end for a non technical user. Gradio is used here since it needs minimal setup and renders directly inside the notebook or a shareable link.

In [ ]:
!pip install -q gradio

In [ ]:
import gradio as gr

def gradio_wrapper(image, description):
    if image is not None:
        result = ecosort_assistant(image_path=image)
    elif description and description.strip():
        result = ecosort_assistant(description=description)
    else:
        return "Please provide either an image or a text description."

    return (
        f"Predicted category, {result['predicted_category']} "
        f"(confidence, {result['confidence']:.2f})\n\n"
        f"Disposal instructions, {result['disposal_instructions']}\n\n"
        f"Source, {', '.join(result['policy_sources'])}"
    )

demo = gr.Interface(
    fn=gradio_wrapper,
    inputs=[
        gr.Image(type='filepath', label='Upload a photo of the item'),
        gr.Textbox(label='Or describe the item')
    ],
    outputs=gr.Textbox(label='EcoSort Result'),
    title='EcoSort Waste Management Assistant',
    description='Upload a photo or describe a waste item to get Metro City disposal instructions.'
)

demo.launch(share=True)

`[FILL IN AFTER KAGGLE RUN]` Note whether the interface behaves correctly for both input types, include a screenshot of the running interface if the notebook is being submitted as a static export.

# Final Reflections

**What worked well**

`[FILL IN AFTER KAGGLE RUN]` Note which component performed best against its Section 1 success criteria, and why.

**Challenges encountered**

The RealWaste image set initially arrived with 3,146 of 4,752 files unreadable, concentrated almost entirely in 6 of the 9 classes, traced back to an interrupted zip extraction rather than scattered individual corruption. A clean re download resolved it, a reminder to verify a dataset's integrity at the start of Data Preparation rather than assuming a download completed correctly.

The RAG system initially used open, unfiltered semantic search across all 14 policy documents regardless of the already known category. This produced two real grounding failures during testing, a broken glass query returning the wrong subsection of the correct document, and a batteries query missing the correct document entirely. The fix was structural, restricting retrieval to only the documents covering the already classified category, `retrieve_policy_chunks_for_category`, rather than relying on open semantic search to find the right document from scratch. This was paired with upgrading the generation model from flan t5 small to flan t5 base and switching decoding from greedy to beam search. `[FILL IN AFTER KAGGLE RUN]` Confirm whether this fix fully resolved both grounding failures once retested, and note if any residual issue remains.

`[FILL IN AFTER KAGGLE RUN]` Note any other modelling challenges once training completes, class imbalance effects that persisted despite weighting, or interface issues.

**What would improve with more time**

Compare MobileNetV2 against EfficientNetB0 directly rather than committing to a single CNN architecture. Expand the RAG evaluation with a larger, more adversarial set of test queries across all 9 categories, not just the 3 used in Section 11, to stress test retrieval accuracy more thoroughly. Replace the TF IDF and classical ML text pipeline with a fine tuned small transformer, worth testing whether it meaningfully outperforms TF IDF given how short these descriptions already are.

**Alignment with Business Understanding**

`[FILL IN AFTER KAGGLE RUN]` Revisit the 4 objectives and success criteria from Section 1 and state plainly whether each was met, partially met, or not met, based on the actual Section 8, 10, and 12 results.